# HW1 Zhenyang Cui

- What: Download, manipulate and merge two or more datasets
- How: Create functions in a python file (src/Zhenyang_Cui/...)
- Show: Show how to use these python files in a notebook

# 1. Research Question

- How does the elasticity of consumption with respect to different types of household wealth differ?
- Specifically, what are the differences in elasticity between housing wealth and stock market wealth in Japan?

# 2. Method

## 2.1 Data Sources

- **Consumption**: Household consumption expenditure (from Family Income and Expenditure Survey - 家計調査)
- **Housing Wealth**: Fixed assets (housing component) from National Accounts (SNA)
- **Stock Market Wealth**: Household financial assets (specifically "stocks") from National Accounts (SNA)
- **Income**: Compensation of employees from National Accounts (SNA)

# 2.2 Empirical Strategy

- Convert all series to:
  - **Real terms** (adjusted for inflation)
  - **Per capita** (divided by total population)

- Conduct a regression based on the following model:

$$
\log C_{t} = \alpha + \beta \log S_{t} + \gamma \log H_{t} + \phi \log Y_{t} + \epsilon_{t}
$$

- Where:

\begin{array}{ll}
C_{t} & : \text{Household consumption} \\
S_{t} & : \text{Stock market wealth} \\
H_{t} & : \text{Housing wealth} \\
Y_{t} & : \text{Compensation of employees}
\end{array}


## 2.3 Main Focus

- Estimate and compare:

\begin{array}{ll}
\gamma & : \text{Elasticity of consumption with respect to housing wealth} \\
\beta & : \text{Elasticity of consumption with respect to stock market wealth}
\end{array}


# 3. Goal

- Understand the relative importance of housing and stock market wealth in driving household consumption behavior in Japan.


# 4.1 Download data

- Download household consumption data.

In [ ]:
import os
import requests

# Set URL and destination path
url = "https://www.esri.cao.go.jp/jp/sna/data/data_list/kakuhou/files/2023/tables/2023ffm1n_jp.xlsx"
save_dir = "../../data/raw_data/japan/"
save_name = "household_consumption.xlsx"
save_path = os.path.join(save_dir, save_name)

# Make sure the save directory exists
os.makedirs(save_dir, exist_ok=True)

# Download and save
response = requests.get(url)
with open(save_path, "wb") as f:
    f.write(response.content)

print(f"File successfully downloaded to: {save_path}")


- Download stock market wealth data.
- Download houseing wealth data.

In [ ]:
import os
import requests

# Create directory if not exists
os.makedirs("data/raw data/japan", exist_ok=True)

# Download URL
url = "https://www.esri.cao.go.jp/jp/sna/data/data_list/kakuhou/files/2023/tables/2023si4_jp.xlsx"

# File path to save
save_path = "data/raw data/japan/household_stock.xlsx"

# Download the file
response = requests.get(url)
with open(save_path, 'wb') as f:
    f.write(response.content)

print("Download completed and saved as 'household_stock.xlsx'.")


- Download income data (compensation of employees).

In [ ]:
import os
import pandas as pd

# Create directory if not exists
os.makedirs("data/raw data/japan", exist_ok=True)

# Download file
url = "https://www.esri.cao.go.jp/jp/sna/data/data_list/kakuhou/files/2023/tables/2023ffm2_jp.xlsx"
save_path = "data/raw data/japan/employment_income.xlsx"
df = pd.read_excel(url)
df.to_excel(save_path, index=False)

print("Employment income data downloaded and saved!")


## 4.2 Manipulate Data

- Manipulate household consumption data.

In [ ]:
import sys
sys.path.append("../../src")  # Allow importing from src/

from data_function.extract_household_consumption import extract_household_consumption

# Extract data
consumption_data = extract_household_consumption(save_path)

# Save processed data
processed_dir = "../../data/processing_data/japan/"
os.makedirs(processed_dir, exist_ok=True)

processed_path = os.path.join(processed_dir, "household_consumption_processed.csv")
consumption_data.to_csv(processed_path, index=False)

print(f"Processed data saved to: {processed_path}")


- Manipulate
  - stock market wealth data.
  - houseing wealth data.

In [ ]:
from src.extract_housing_and_stock import extract_housing_and_stock
import os

# Create directory if not exists
os.makedirs("data/processing data/japan", exist_ok=True)

# Extract data
file_path = "data/raw data/japan/household_stock.xlsx"
processed_data = extract_housing_and_stock(file_path)

# Save as CSV
processed_data.to_csv("data/processing data/japan/processed_household_stock.csv", index=False)

print("Processed data saved as CSV!")

- Manipulate income data (compensation of employees).

In [ ]:
from src.data_function.extract_employment_income import extract_employment_income

# Create directory if not exists
os.makedirs("data/processing data/japan", exist_ok=True)

# Extract data
file_path = "data/raw data/japan/employment_income.xlsx"
processed_employment_income = extract_employment_income(file_path)

# Save as CSV
processed_employment_income.to_csv("data/processing data/japan/processed_employment_income.csv", index=False)

print("Processed employment income data saved as CSV!")

## 4.3 Merge Data

In [ ]:
# Section 1: Load all processed CSVs

import pandas as pd
import os

# Define file paths
raw_path = "../../data/processed/japan/"  # Processed CSV files folder
clean_path = "../../data/clean/"           # Save merged data here

csv_files = {
    "household_consumption": os.path.join(raw_path, "processed_household_consumption.csv"),
    "housing_stock_wealth": os.path.join(raw_path, "processed_household_stock.csv"),
    "employee_income": os.path.join(raw_path, "processed_employment_income.csv"),
}

# Read each dataset
consumption = pd.read_csv(csv_files["household_consumption"])
housing_stock = pd.read_csv(csv_files["housing_stock_wealth"])
employee_income = pd.read_csv(csv_files["employee_income"])

# Check
print(consumption.head())
print(housing_stock.head())
print(employee_income.head())

# Merge on 'year'
merged_data = consumption.merge(
    housing_stock, on="year", how="inner"
).merge(
    employee_income, on="year", how="inner"
)

# Check the merged dataset
print(merged_data.head())

# Confirm columns
print(merged_data.columns)

# Create the clean directory if not exists
os.makedirs(clean_path, exist_ok=True)

# Save
merged_data.to_csv(os.path.join(clean_path, "merged_macro_data.csv"), index=False)

print("Merged dataset saved successfully!")


## 5. Descriptive Statistics

In [ ]:
# Import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load merged data
merged_data = pd.read_csv("../../data/clean/merged_macro_data.csv")

# Display basic info
print(merged_data.info())

# Descriptive statistics
print(merged_data.describe())

# Correlation matrix
corr_matrix = merged_data.corr()
print(corr_matrix)

# Heatmap of correlation
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix of Macroeconomic Variables")
plt.show()


## 6. Visualization

In [ ]:
# Line plots over time
plt.figure(figsize=(12, 8))

# Consumption
plt.plot(merged_data["year"], merged_data["consumption"], label="Consumption")

# Housing Wealth
plt.plot(merged_data["year"], merged_data["fixed_asset"], label="Housing Wealth")

# Stock Wealth
plt.plot(merged_data["year"], merged_data["stock_asset"], label="Stock Wealth")

# Employee Income
plt.plot(merged_data["year"], merged_data["employee_income"], label="Employee Income")

plt.xlabel("Year")
plt.ylabel("Amount")
plt.title("Macroeconomic Indicators Over Time")
plt.legend()
plt.grid(True)
plt.show()


## 7. Regression

In [ ]:
# Create log-transformed variables
merged_data["log_consumption"] = np.log(merged_data["consumption"])
merged_data["log_fixed_asset"] = np.log(merged_data["fixed_asset"])
merged_data["log_stock_asset"] = np.log(merged_data["stock_asset"])
merged_data["log_employee_income"] = np.log(merged_data["employee_income"])

# Check
print(merged_data[["log_consumption", "log_fixed_asset", "log_stock_asset", "log_employee_income"]].head())


In [ ]:
import statsmodels.api as sm

# Define independent variables
X = merged_data[["log_fixed_asset", "log_stock_asset", "log_employee_income"]]
X = sm.add_constant(X)

# Define dependent variable
y = merged_data["log_consumption"]

# OLS regression
model = sm.OLS(y, X).fit()

# Result summary
print(model.summary())
